<a href="https://colab.research.google.com/github/LeeSeongJinn/Doosan_Rokey_bootcamp/blob/main/2_Applied_AI/Day14/1_ViT_Beans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 동일한 CUDA 버전(cu124)의 PyTorch 생태계 패키지를 명시적으로 설치
!pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 2. 나머지 의존성 패키지 설치
!pip install -q -U "datasets>=3.0.1" "transformers>=4.45.2" "accelerate>=1.0.1" "evaluate>=0.4.2"

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__, "| Datasets:", datasets.__version__)

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download-r2.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (28 kB)
  Using cached https://download-r2.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (6.1 kB)
  Using cached https://download-r2.pytorch.org/whl/cu124/torchaudio-2.6.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (6.6 kB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 152.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 223.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

^C
PyTorch: 2.6.0+cu124 | CUDA: True
Transformers: 5.16.1 | Datasets: 5.0.1


In [ ]:
!pip install evaluate
!pip install accelerate -U

  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
Using cached fsspec-2026.6.0-py3-none-any.whl (203 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.7.0
    Uninstalling fsspec-2026.7.0:
      Successfully uninstalled fsspec-2026.7.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.6.0 which is incompatible.


  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.15.0-py3-none-any.whl (394 kB)
^C


In [ ]:
import torch, transformers, datasets, evaluate
import numpy as np

In [ ]:
from datasets import load_dataset
beans = load_dataset("AI-Lab-Makerere/beans")
print(beans)

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})


In [ ]:
beans['train'].features

{'image_file_path': Value('string'),
 'image': Image(mode=None, decode=True),
 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'])}

In [ ]:
beans['train'].features

{'image_file_path': Value('string'),
 'image': Image(mode=None, decode=True),
 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'])}

In [ ]:
key = 'labels' if 'labels' in beans['train'].features else 'label'
beans['train'].features[key].names

['angular_leaf_spot', 'bean_rust', 'healthy']

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from transformers import AutoImageProcessor, ViTForImageClassification
import torch

MODEL = "google/vit-base-patch16-224"

processor = AutoImageProcessor.from_pretrained(MODEL)

# 라벨 매핑 설정
key = 'labels' if 'labels' in beans['train'].features else 'label'
names = beans['train'].features[key].names
id2label = {i: n for i, n in enumerate(names)}
label2id = {n: i for i, n in enumerate(names)}

model = ViTForImageClassification.from_pretrained(
    MODEL,
    num_labels=len(names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
model.classifier.out_features # 분류기 크기: 3개

3

In [ ]:
import os

def transform(ex):
    # 입력이 PIL 이미지 리스트 (batched=True)
    # processor 가 알아서 resize, 정규화 수행, tensor 변환
    inputs = processor(images=ex['image'], return_tensors='pt')
    # 이미지 리스트를 프로세서에 전달

    # 결과 저장
    ex['pixel_values'] = inputs['pixel_values']
    # pixel_values : 4차원 텐서로 출력 (b, c, h, w)
    # (입력된 이미지의) 픽셀 값만 추출 >> 배치에 추가
    return ex

# 먼저 map 전처리 수행 (병렬 처리 추가)
# remove_columns 사용, 원본 'image' 컬럼 제거 (메모리 절약)

beans = beans.map(
        transform,
        batched=True,
        remove_columns=['image'], # 학습에 필요 없는 원본 이미지 컬럼 삭제
    )


# format 설정
# Hugging Face Trainer 쓰면 이 부분 생략 >> 대신, DataCollater 가 처리하게 함
beans.set_format('torch')

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

In [ ]:
def keep(split):
    cols = ['pixel_values', key]
    # pixel_values 이미지 데이터, key: 라벨
    # 즉, cols 는 이미지데이터, 라벨 을 남기겠다.
    # split : train, val, test
    return beans[split].remove_columns([c for c in beans[split].column_names if c not in cols])
    # 지울 컬럼리스트 = 전체 컬럼 - 남길 컬럼

In [ ]:
train, val, test = keep('train'), keep('validation'), keep('test')

In [ ]:
from transformers import TrainingArguments, Trainer, DefaultDataCollator
import evaluate
import numpy as np
import torch

# 정확도 지표 로드
acc = evaluate.load("accuracy")

# 평가 계산 함수 정의
def metrics(p):
    predictions, labels = p
    pred = np.argmax(predictions, axis=1)
    return {'accuracy': acc.compute(predictions=pred, references=labels)['accuracy']}

# TrainingArguments 설정
args = TrainingArguments(
        output_dir='/content/vit_beans',
        eval_strategy='epoch',
        save_strategy='epoch',
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        learning_rate=2e-5,
        fp16=torch.cuda.is_available(),
        report_to='none',
        remove_unused_columns=False    # 이미지 데이터셋 컬럼 유지 위해 권장
    )

# Trainer 초기화
# tokenizer=processor >> 이것 대신에 data_collater 명시하는 게 정석임
trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train,
        eval_dataset=val,
        data_collator=DefaultDataCollator(), # 텐서 배치를 위한 Collater
        compute_metrics=metrics
    )

# DefaultDataCollator()
# 각 개별 샘플들을 하나씩 깔끔하게 배치(batch) tensor 로 묶어주는 역할

# 학습 시작
trainer.train()

print(trainer.evaluate(test))

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.098660,0.962406
2,No log,0.047961,0.992481
3,No log,0.040924,0.992481


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
No log,0.076875,3,0.976562


{'eval_loss': 0.0768754780292511, 'eval_accuracy': 0.9765625}


In [ ]:
import torch

for i in [0,1]:
    ex = beans['test'][i]

    # 입력 데이터 처리
    input_tensor = ex['pixel_values'].clone().detach()
    inputs = input_tensor.unsqueeze(0).to(model.device)

    # 모델 예측
    with torch.no_grad():
        logits = model(inputs).logits
        pred = logits.argmax(-1).item() # 정수변환

    # 정답 라벨 가져오기
    label_key = 'labels' if 'labels' in ex else 'label'

    true_label_id = ex[label_key].item() #.item() 사용하면 tensor(0) >> 0(정수)

    # 결과 출력
    print(f'[{i} 예측: {model.config.id2label[pred]} | {model.config.id2label[true_label_id]}]')



[0 예측: angular_leaf_spot | angular_leaf_spot]
[1 예측: angular_leaf_spot | angular_leaf_spot]


In [ ]:
# eos